# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [61]:
# Write your code below.
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [62]:
import os
from glob import glob
import pandas as pd
import random
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [63]:

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True) 


In [64]:
dd_px = dd.read_parquet(parquet_files).set_index("ticker")

In [65]:
dd_px


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
npartitions=60,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32
ALDX,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [66]:
# Write your code below.
dd_shift = (
    dd_px
        .groupby('ticker', group_keys=False)
        .apply(
            lambda x: x.sort_values('Date', ascending = True)
                       .assign(
                           Close_lag_1 = x['Close'].shift(1),
                           Adj_Close_lag_1 = x['Adj Close'].shift(1)
                        ),        
            meta = pd.DataFrame(data ={
                    'Date': 'datetime64[ns]',
                    'Open': 'f8',
                    'High': 'f8',
                    'Low': 'f8',
                    'Close': 'f8',
                    'Adj Close': 'f8',
                    'Volume': 'i8',
                    'source': 'object',
                    'Year': 'int32',
                    'Close_lag_1': 'f8',
                    'Adj_Close_lag_1':'f8'},
                    index = pd.Index([], dtype=pd.StringDtype(), name='ticker'))
        )
)

dd_feat = dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1,
    Hi_Lo_range = lambda x: x['High'] - x['Low']
)

In [67]:
# inspect dask data for added columns
dd_feat.head()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,Hi_Lo_range
ticker,,,,,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN,NaN,0.29
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,11.404394,-0.010547,0.25
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,11.284108,-0.000666,0.46
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,11.276587,-0.009333,0.27
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,11.171341,0.006057,0.30


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [76]:
# Write your code below.
df_feat = dd_feat.compute()   # execute Dask, since Dask date was laziliy  --> now df_feat is a panda data frame

df_feat = df_feat.reset_index() # reset the index so the tickers become a column in Panda data frame


# 10-day moving average of returns
df_feat['Returns_10day_Moving_Average'] = df_feat.groupby('ticker')['Returns'].transform(lambda x: x.rolling(10).mean())



In [79]:
# Inspect the results
df_feat.head(15)

,ticker,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,Hi_Lo_range,Returns_10day_Moving_Average
0,ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN,NaN,0.29,NaN
1,ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,11.404394,-0.010547,0.25,NaN
2,ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,11.284108,-0.000666,0.46,NaN
3,ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,11.276587,-0.009333,0.27,NaN
4,ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,11.171341,0.006057,0.30,NaN
5,ACN,2001-07-26,14.95,14.99,14.50,14.50,10.900705,6335300.0,ACN.csv,2001,14.95,11.238999,-0.030100,0.49,NaN
6,ACN,2001-07-27,14.51,14.59,14.50,14.51,10.908223,3524000.0,ACN.csv,2001,14.50,10.900705,0.000690,0.09,NaN
7,ACN,2001-07-30,14.50,14.78,14.50,14.70,11.051059,3654300.0,ACN.csv,2001,14.51,10.908223,0.013094,0.28,NaN
8,ACN,2001-07-31,14.71,15.01,14.60,14.96,11.246520,1429000.0,ACN.csv,2001,14.70,11.051059,0.017687,0.41,NaN
9,ACN,2001-08-01,15.00,15.50,14.90,15.50,11.652478,2087900.0,ACN.csv,2001,14.96,11.246520,0.036096,0.60,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
ANSWER: No, it was not necesaryy but it was more straight forward and simpler to calculate the moving averaga in Panda using .rolling() method for the size of this database. 
It is also the most readable approach --> Simplicity and Readability. Note that for this assignmen, data size was not too big. We could also calculate moving average in Dask too but would be more complex since we need to combine rolling window with groupby to report per ticker.

+ Would it have been better to do it in Dask? Why?
ANSWER: Not for this assignment, with current data, because the dataset fits comfortably in memory, and pandas is simpler and easier to use. For much larger datasets, Dask would be better because it can handle data that doesn’t fit in memory and parallelize computations. If the database was huge or could become huge in future runs, the step to compute Dask dataframe, which was needed before converting too panda, could exceed memory limit. In such case, we would use Dask that provides out-of-core calculation. Dask is better for production-scale ML Pipleline / ML system to scale up. Note: in case of 10-day moving average per ticker, it will be tricky to handle it. Dask has map_partion and rolling for an individual partition but needs more expertise to do rolling window across group. Such complex work would pay off for a good reliable, scale-able ML system.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.